# Data Analytics — Task 1
## Data Quality and Exploratory Analysis

**Objective:** Audit and clean the supplied customer churn dataset, document quality decisions and validation checks, calculate summary statistics, and inspect meaningful distributions.

This notebook directly addresses the four required Task-1 steps:
1. Profile columns, data types, missing values, outliers, and duplicates.
2. Clean the data with documented assumptions and validation checks.
3. Calculate summary statistics and inspect meaningful distributions.
4. Package the work and explanation for one public/view-only proof link.


## 1. Load and profile the dataset
The supplied dataset contains customer account and churn information. The profiling stage checks shape, columns, data types, missing values, uniqueness and categorical values.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
INPUT = ROOT / "data" / "customer_churn_dataset.csv"
df_raw = pd.read_csv(INPUT)

print("Shape:", df_raw.shape)
print("\nColumns:", df_raw.columns.tolist())
display(df_raw.head())
display(df_raw.dtypes.to_frame("Data Type"))
display(df_raw.isna().sum().to_frame("Missing Values"))
print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Duplicate CustomerIDs:", df_raw["CustomerID"].duplicated().sum())


## 2. Data-quality findings

The source data contains:
- 1,003 rows and 11 columns.
- 4 missing Age values.
- 3 missing MonthlyCharges values.
- 2 missing ContractType values.
- 3 exact duplicate rows / duplicate CustomerIDs.
- A few categorical values with leading/trailing spaces or case variants.
- IQR-based numerical outliers, including unusually high MonthlyCharges and SupportCalls.

Outliers are **flagged rather than automatically deleted**, because an IQR rule identifies unusual values but does not by itself prove that a record is invalid.


In [ ]:
# Category normalization
df = df_raw.copy()
cat_cols = ["CustomerID", "Gender", "ContractType", "PaymentMethod", "InternetService", "Churn"]
for c in cat_cols:
    df[c] = df[c].apply(lambda x: x.strip() if isinstance(x, str) else x)

df["Gender"] = df["Gender"].replace({"female":"Female", "Male ":"Male", " female ":"Female", " female":"Female"})
df["ContractType"] = df["ContractType"].replace({"month-to-month":"Month-to-month", " month-to-month ":"Month-to-month", "One year ":"One year"})

num_cols = ["Age","TenureMonths","MonthlyCharges","TotalCharges","SupportCalls"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Remove exact duplicates, then duplicate CustomerIDs
df = df.drop_duplicates().copy()
df = df.drop_duplicates(subset=["CustomerID"], keep="first").copy()

# Missing values: median for numeric, mode for categorical
df["Age"] = df["Age"].fillna(df["Age"].median())
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(df["MonthlyCharges"].median())
df["ContractType"] = df["ContractType"].fillna(df["ContractType"].mode().iloc[0])

df["MonthlyCharges"] = df["MonthlyCharges"].round(2)
df["TotalCharges"] = df["TotalCharges"].round(2)

print("Rows after cleaning:", len(df))
print("Missing cells after cleaning:", df.isna().sum().sum())
display(df.head())


### Cleaning assumptions

- Trim whitespace from text categories.
- Standardize obvious category variants such as `female` → `Female`.
- Remove exact duplicate rows.
- For duplicate CustomerIDs, retain the first record after exact-duplicate removal to keep one customer record per ID.
- Impute missing Age and MonthlyCharges with the cleaned-data median.
- Impute missing ContractType with the mode.
- Round monetary fields to two decimals.
- Do not delete IQR outliers automatically; flag them for review.


In [ ]:
# IQR outlier report
outlier_rows = []
for c in num_cols:
    s = pd.to_numeric(df_raw[c], errors="coerce").dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    count = ((s < lower) | (s > upper)).sum()
    outlier_rows.append([c, q1, q3, iqr, lower, upper, count, s.min(), s.max()])
outliers = pd.DataFrame(outlier_rows, columns=["Column","Q1","Q3","IQR","Lower","Upper","Outlier Count","Min","Max"])
display(outliers.round(2))


## 3. Validation checks

In [ ]:
checks = pd.DataFrame({
    "Check": [
        "No missing values after cleaning",
        "CustomerID unique",
        "Age between 18 and 100",
        "TenureMonths between 0 and 120",
        "MonthlyCharges non-negative",
        "TotalCharges non-negative",
        "SupportCalls non-negative",
        "Churn only Yes/No"
    ],
    "Result": [
        not df.isna().any().any(),
        not df["CustomerID"].duplicated().any(),
        df["Age"].between(18,100).all(),
        df["TenureMonths"].between(0,120).all(),
        df["MonthlyCharges"].ge(0).all(),
        df["TotalCharges"].ge(0).all(),
        df["SupportCalls"].ge(0).all(),
        df["Churn"].isin(["Yes","No"]).all()
    ]
})
display(checks)


## 4. Summary statistics

In [ ]:
summary = df[num_cols].describe().T.round(2)
display(summary)


## 5. Meaningful distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["Churn"].value_counts().reindex(["No","Yes"]).plot(kind="bar", ax=axes[0])
axes[0].set_title("Customer Churn Distribution")
axes[0].set_xlabel("Churn"); axes[0].set_ylabel("Customers"); axes[0].tick_params(axis="x", rotation=0)

df["ContractType"].value_counts().reindex(["Month-to-month","One year","Two year"]).plot(kind="bar", ax=axes[1])
axes[1].set_title("Contract Type Distribution")
axes[1].set_xlabel("Contract Type"); axes[1].set_ylabel("Customers"); axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df.boxplot(column="Age", ax=axes[0])
axes[0].set_title("Age")
df.boxplot(column="MonthlyCharges", ax=axes[1])
axes[1].set_title("Monthly Charges")
df.boxplot(column="SupportCalls", ax=axes[2])
axes[2].set_title("Support Calls")
plt.tight_layout()
plt.show()


## 6. Key observations

- The dataset contains 1,003 raw rows and 11 columns.
- After removing duplicate customer records, the cleaned dataset contains 1,000 unique customers.
- Missing values were limited to Age, MonthlyCharges and ContractType and were treated with documented median/mode rules.
- ContractType is concentrated in Month-to-month contracts, followed by One year and Two year.
- Churn is present in the dataset and is suitable for exploratory distribution analysis.
- IQR flags unusually high values in some numeric fields, especially MonthlyCharges and SupportCalls. These were retained because they may represent genuine customer records.
- Findings are descriptive for this supplied dataset; they should not be treated as causal conclusions.
